- _exponent._bronze_allscripts_tw_works_vw.dbo_patient_member
- _exponent._bronze_allscripts_tw_works_vw.dbo_person
- _exponent._bronze_allscripts_tw_works_vw.dbo_person_other
- _exponent._bronze_allscripts_tw_works_vw.dbo_ethnicity_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_race_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_sex_de


In [0]:
source = 'allscripts_tw'

# Transformation

In [0]:
%sql
SELECT 
* 
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_ethnicity_de
WHERE IsInactiveFLAG = 'N'
-- WHERE ID = 4123341
--  LIMIT 10 

In [0]:
%sql
SELECT 
  dbo_person.*,
  dbo_race_de.entrycode,
  dbo_sex_de.entryname
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_person
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person_other
ON dbo_person_other.id = dbo_person.id 
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_race_de
ON dbo_race_de.id = dbo_person_other.racede
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_sex_de
ON dbo_sex_de.id = dbo_person.sexde
-- LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_race_de
-- ON dbo_race_de.id = dbo_person.racede
WHERE dbo_person.isinactiveflag = 'N'
AND dbo_person.id = 4123341
LIMIT 100

In [0]:
%sql
SELECT
'allscipts_tw' AS source_system,
'dbo_ethnicity_de' AS source_table,
'EntryName' AS source_field,
dbo_ethnicity_de.id AS source_id,
dbo_ethnicity_de.entryname AS source_value,
concept.concept_id AS omop_concept_id,
concept.concept_name AS omop_concept_name,
CASE WHEN dbo_ethnicity_de.isinactiveflag = 'N' THEN 1 ELSE 0 END AS active_flag
FROM _exponent.omop.concept
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_ethnicity_de
ON LOWER(concept.concept_code) = LOWER(dbo_ethnicity_de.entryname)
WHERE domain_id = 'Ethnicity'
-- AND standard_concept = 'S'


In [0]:
bronze_person = spark.sql(f'''
SELECT 
  dbo_person.id AS person_id,
  CASE WHEN LOWER(dbo_sex_de.entryname) = 'male' THEN 8507
       WHEN LOWER(dbo_sex_de.entryname) = 'female'  THEN 8532
       ELSE 8551 END AS gender_concept_id,
  YEAR(dbo_person.dateofbirth) AS year_of_birth,
  MONTH(dbo_person.dateofbirth) AS month_of_birth,
  DAY(dbo_person.dateofbirth) AS day_of_birth,
  dbo_person.dateofbirth AS birth_datetime,
  CONCAT('{source}', ' | ', dbo_person.id) AS person_source_value,
  dbo_sex_de.entryname AS gender_source_value,
  0 AS gender_source_concept_id

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_person
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_sex_de
ON dbo_sex_de.id = dbo_person.sexde
WHERE dbo_person.isinactiveflag = 'N'
AND dbo_person.id = 4123341
LIMIT 100''')
display(bronze_person)